# FleetRoute V7 — Lightweight Targeted Recovery Benchmark (AUDIT4)

This notebook reruns **only the 18 V7 rows that were terminated by the parent wrapper's post-deadline watchdog**. It does not rerun already-valid rows and does not run OR-Tools.

**Frozen contracts:** six exact public instances, exact STRICT_FAIR Legacy/V3.1 sources, exact seeds, 6 workers, unchanged hard algorithmic budgets, identical command semantics, and the same independent public-instance evaluator. The longer wrapper grace exists only to let the deadline-censored solver ledger serialize and exit; recovery runtime is explicitly non-comparable to the parent batch.

AUDIT4 additionally embeds a compact parent-invalid-18 provenance receipt, verifies all 18 semantic commands before any run, independently revalidates every public reference route, enforces distance-only scoring with route count diagnostic, and aggregates objective values from valid recoveries only.


In [ ]:
# @title 1/3 — Extract six frozen targets, verify contracts, compile Legacy + V3.1
from pathlib import Path
import base64, zipfile, hashlib, json, csv, math, re, collections, subprocess, platform, os, io, shutil

ROOT=Path('/content/vrp_v7_target_recovery_audit4')
if ROOT.exists():
    if (ROOT/'RECOVERY_ATTEMPT_STARTED.lock').exists():
        raise RuntimeError('A recovery experiment already started in this runtime. Restart the runtime before creating a new experiment.')
    shutil.rmtree(ROOT)
DATA=ROOT/'data'; RAW=ROOT/'public_raw'; SRC=ROOT/'sources'; BIN=ROOT/'bin'; RUNS=ROOT/'runs'; RESULTS=ROOT/'results'
for p in (DATA,RAW,SRC,BIN,RUNS,RESULTS): p.mkdir(parents=True,exist_ok=True)

PAYLOAD_SHA256='512b95586c21713b6555fb20218b9880ff95c59c69acd20fa4f293c6df6e0ac3'
PAYLOAD_B64='...'
EXPECTED_FILE_SHA256={'data/CVRPLIB-X_X-n200-k36.txt': '4b9c7fee7e0b9e4d8287ab9750c513704bffc65ad1c02552e8b303a93f68e211', 'data/CVRPLIB-X_X-n200-k36_coords.csv': 'fb1d2bd473150fa541747d225b942e1e3e924a5606691dfe359762698687bfd9', 'data/CVRPLIB-X_X-n303-k21.txt': '48a48d46a7ee983b48fe3d8ec1b53d1349f7048287ad756d02b5691b5ff99e62', 'data/CVRPLIB-X_X-n303-k21_coords.csv': '2ce8d4c5e9836555d3c5a3743d0cee98138c7c543dc0bd53c10f70aeb2871313', 'data/Gehring-Homberger-200_C2_2_1.txt': 'fa66d2f0ac6c4bff54b0db730adaaab2de6176c73b8d6f96626cc5d3a3a196c7', 'data/Gehring-Homberger-200_C2_2_1_coords.csv': '36a67b448ef731ecf8d63f44e82d79d22cc190274db06d2418c537032e589e4f', 'data/Gehring-Homberger-200_C2_2_1_v31_tw_feasible_seed.csv': 'cdbcf9b1265ee43ab403bdbd74058611cc4a4cf64f2ce0118ed3d282de82fc1d', 'data/Gehring-Homberger-200_R1_2_1.txt': '9712f9fea6215da3823102c27dabc1d89c16e162fdde38ec7eea9da8b897fc3d', 'data/Gehring-Homberger-200_R1_2_1_coords.csv': 'fa65abdaddd2765e389d4715638ff3bf84a8e2dfad7500203ea682c687f7a6d3', 'data/Gehring-Homberger-200_R1_2_1_v31_tw_feasible_seed.csv': 'cdbcf9b1265ee43ab403bdbd74058611cc4a4cf64f2ce0118ed3d282de82fc1d', 'data/Gehring-Homberger-200_R2_2_1.txt': '3f03e91f8098975faa7bc1b69f7ed14eea28c09cd3071ab0094a54eca729b5c8', 'data/Gehring-Homberger-200_R2_2_1_coords.csv': 'ad074743bff728a6b496d6a73cdf2f1473ee59fab500326a3317a1da4d8b1406', 'data/Gehring-Homberger-200_R2_2_1_v31_tw_feasible_seed.csv': 'cdbcf9b1265ee43ab403bdbd74058611cc4a4cf64f2ce0118ed3d282de82fc1d', 'data/Gehring-Homberger-200_RC2_2_1.txt': '97db2e654528a323bdc506ff99e65e1876fed886c6212fc3a86e5f573df4f051', 'data/Gehring-Homberger-200_RC2_2_1_coords.csv': '9cbce87a60c7c2ed1b07df91f2268725df85cd5fc1111bb51e4a345eedfa9b83', 'data/Gehring-Homberger-200_RC2_2_1_v31_tw_feasible_seed.csv': 'cdbcf9b1265ee43ab403bdbd74058611cc4a4cf64f2ce0118ed3d282de82fc1d', 'metadata/instance_manifest_subset.json': '03c4b97028818aa245da78125933ea2db44fdaa541448519e713b9f5c5756c3a', 'metadata/recovery_parent_invalid18_receipt.json': '2372398139aceec55ee7676b54752f780fd131d38ab2c3c9a9ab6efcf1821a76', 'public_raw/C2_2_1.sol': 'ef9cc73e64c9d8fad7c23f71af378c673a9ed3f1f034a38a7fcf49144954ec61', 'public_raw/C2_2_1.vrp': '558fe2542e07fa622c9cb8272034b15756ef6718b62ce0d20e75f89005bf5009', 'public_raw/R1_2_1.sol': '0603ad0ac00e614d1cc19bcc01fb86ad873bc1268de54927bc593b1a0c0879c1', 'public_raw/R1_2_1.vrp': 'b45b327caf7dbffd6694273f806463972d95ca31e35372e5f62022a085c3f435', 'public_raw/R2_2_1.sol': '464af983210b063a885ce76a2ab22146b0d85d476de2c31fc7ddb7cef42a0f43', 'public_raw/R2_2_1.vrp': '2206b151f8ca5c8da08f7cbfa62f00b1dcb023432ced0779a6f9c7eb121a5cfc', 'public_raw/RC2_2_1.sol': '1c0293f420e9a21bf66512c91114c41e6a438d1376da8f2e94eeed47074334fa', 'public_raw/RC2_2_1.vrp': 'a5ddcda2f0fd168cfb91108255f97766dafbba363763e325eab29dc2f15bbb62', 'public_raw/X-n200-k36.sol': '3f8afac8b869caffed19cbc3309e6667665d8ee82b59b7606602cd56bd2556b1', 'public_raw/X-n200-k36.vrp': 'c9524bedaf4b9aac542538ca56fffd33ae935f667f3580fbd1d61f1c2c5988f9', 'public_raw/X-n303-k21.sol': '16d9b34b3d4d4047034160972b004ecb8e63daa3ba2d7b9207303386598a85e9', 'public_raw/X-n303-k21.vrp': '8b31998ba5500255b3b2b81bacbe1e5467209439ca8efae9ba099e1190724dfe', 'sources/solver_legacy_STRICT_FAIR_V6.cpp': '6225684b27bdc8c1c484d1576e21f069c5dc2229aaaea7ff7223c7f2e84684cc', 'sources/solver_v31_STRICT_FAIR_V6.cpp': '845014863fe646f96969f775f68851411f0ff0ab4586d487a3ade0098a2e8a22'}
EXPECTED_BENCH_SOURCE_SHA={
    'legacy':'6225684b27bdc8c1c484d1576e21f069c5dc2229aaaea7ff7223c7f2e84684cc',
    'v31':'845014863fe646f96969f775f68851411f0ff0ab4586d487a3ade0098a2e8a22',
}
EXPECTED_ORIGINAL_SOURCE_SHA={
    'legacy':'1f5822cfc49c108616e63e6bc833a854b39eab36079ccb33bcb1ce388690dee7',
    'v31':'2508570e517419860a9b278e3692065316503a0f67efeb2d9ad16ded16de19de',
}

blob=base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(blob).hexdigest()==PAYLOAD_SHA256, 'embedded mini-payload SHA mismatch'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(ROOT)
for rel,expected in EXPECTED_FILE_SHA256.items():
    p=ROOT/rel
    got=hashlib.sha256(p.read_bytes()).hexdigest()
    assert got==expected, f'file hash mismatch: {rel}'

SECTION_NAMES={'NODE_COORD_SECTION','DEMAND_SECTION','TIME_WINDOW_SECTION','DEPOT_SECTION','SERVICE_TIME_SECTION','EDGE_WEIGHT_SECTION','EOF'}
def parse_vrplib(path):
    text=Path(path).read_text(errors='replace'); lines=[ln.strip() for ln in text.splitlines() if ln.strip()]
    header={}; sections=collections.defaultdict(list); section=None
    for ln in lines:
        up=ln.upper()
        if up in SECTION_NAMES:
            section=None if up=='EOF' else up; continue
        if section is None and ':' in ln:
            k,v=ln.split(':',1); header[k.strip().upper()]=v.strip().strip('"'); continue
        if section is not None: sections[section].append(ln)
    dim=int(float(header['DIMENSION'])); cap=float(header['CAPACITY'])
    coords={int(p[0]):(float(p[1]),float(p[2])) for p in (ln.split() for ln in sections['NODE_COORD_SECTION'])}
    demands={int(p[0]):float(p[1]) for p in (ln.split() for ln in sections['DEMAND_SECTION'])}
    tw={int(p[0]):(float(p[1]),float(p[2])) for p in (ln.split() for ln in sections['TIME_WINDOW_SECTION'])}
    service={int(p[0]):float(p[1]) for p in (ln.split() for ln in sections['SERVICE_TIME_SECTION'])}
    const_service=float(header.get('SERVICE_TIME','0') or 0)
    depot=None
    for ln in sections['DEPOT_SECTION']:
        v=int(float(ln.split()[0]))
        if v>=0: depot=v; break
    if depot is None: depot=1
    name=header.get('NAME',Path(path).stem).strip(); typ=header.get('TYPE','CVRP').upper()
    k_explicit=None
    for key in ('VEHICLES','TRUCKS'):
        if key in header:
            try: k_explicit=int(float(header[key])); break
            except: pass
    km=re.search(r'-k(\d+)',name,re.I); k_name=int(km.group(1)) if km else None
    return dict(name=name,type=typ,dimension=dim,capacity=cap,coords=coords,demands=demands,time_windows=tw,
                service=service,const_service=const_service,depot=depot,k_explicit=k_explicit,k_name=k_name,header=header)

def parse_sol(path):
    text=Path(path).read_text(errors='replace')
    pat=re.compile(r'Route\s*#?\s*\d+\s*:\s*(.*?)(?=(?:Route\s*#?\s*\d+\s*:)|(?:\bCost\b\s*:?)|\Z)',re.I|re.S)
    routes=[]
    for m in pat.finditer(text):
        ids=[int(x) for x in re.findall(r'-?\d+',m.group(1).strip())]
        if ids: routes.append(ids)
    cm=re.search(r'\bCost\b\s*:?[ \t]*([-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)',text,re.I)
    return {'routes':routes,'cost_stored':float(cm.group(1)) if cm else None}

def euclid(a,b): return math.hypot(a[0]-b[0],a[1]-b[1])
def euc_round(a,b): return math.floor(euclid(a,b)+0.5)
def euc_dimacs(a,b): return math.floor(euclid(a,b)*10.0+1e-12)/10.0

def parse_routes(text):
    routes=[]
    for line in text.splitlines():
        if 'Path' in line and ':' in line:
            rhs=line.split(':',1)[1].strip()
            if '->' in rhs:
                try: routes.append([int(x.strip()) for x in rhs.split('->')])
                except: pass
    return routes

def active_routes(routes,depot=0): return [r for r in routes if r and any(x!=depot for x in r)]

def validate_euclidean(meta,routes):
    parsed=meta['parsed']; coords={nid-1:xy for nid,xy in parsed['coords'].items()}; depot=0
    demands={nid-1:v for nid,v in parsed['demands'].items()}; tw={nid-1:v for nid,v in parsed['time_windows'].items()}
    service={nid-1:parsed['service'].get(nid,parsed['const_service']) for nid in parsed['coords']}
    dist=euc_dimacs if meta['truncate_1_decimal'] else (euc_round if meta['round_distance'] else euclid)
    routes=active_routes(routes,depot); expected=set(range(1,parsed['dimension'])); visits=[]; total=late=max_load=0.0
    depot_return_lateness=0.0; bad=[]; overload=[]; unknown=[]; internal_depot=[]
    dep_e,dep_l=tw.get(depot,(0.0,1e12))
    for ri,r in enumerate(routes):
        if r[0]!=depot or r[-1]!=depot: bad.append(ri)
        if any(x==depot for x in r[1:-1]): internal_depot.append(ri)
        load=0.0; clock=max(meta['route_start_s'],dep_e) if meta['use_time_window'] else meta['route_start_s']
        if meta['use_time_window'] and clock>dep_l:
            q=clock-dep_l; late+=q; depot_return_lateness+=q
        for a,b in zip(r,r[1:]):
            if a not in coords or b not in coords: unknown.append((ri,a,b)); continue
            d=dist(coords[a],coords[b]); total+=d
            if meta['use_time_window']: clock+=d
            if b==depot:
                if meta['use_time_window']:
                    q=max(0.0,clock-dep_l); late+=q; depot_return_lateness+=q
                continue
            visits.append(b); load+=demands.get(b,0.0)
            if meta['use_time_window']:
                e,l=tw.get(b,(0.0,1e12)); start=max(clock,e); late+=max(0.0,start-l); clock=start+service.get(b,0.0)
        max_load=max(max_load,load)
        if load>meta['capacity']+1e-7: overload.append((ri,load))
    cnt=collections.Counter(visits); duplicates=sorted([x for x,c in cnt.items() if c>1]); seen=set(visits)
    missing=sorted(expected-seen); extra=sorted(seen-expected)
    valid=(len(routes)<=meta['K'] and not bad and not internal_depot and not overload and not unknown and not duplicates and not missing and not extra and late<=1e-6)
    return dict(valid=valid,common_distance=total,used_routes=len(routes),lateness=late,depot_return_lateness=depot_return_lateness,
                max_load=max_load,duplicates=duplicates,missing=missing,extra=extra,bad_depot_routes=bad,
                internal_depot_routes=internal_depot,overload_routes=overload,unreachable=unknown)

def reference_common_cost(parsed,sol,tw_mode):
    coords={nid-1:xy for nid,xy in parsed['coords'].items()}; dist=euc_dimacs if tw_mode else euc_round; total=0.0
    for rr in (r for r in sol['routes'] if r):
        r=[0]+rr+[0]; total+=sum(dist(coords[a],coords[b]) for a,b in zip(r,r[1:]))
    return total

def parse_native_euclidean_input(path):
    toks=Path(path).read_text().split(); i=0
    N=int(toks[i]); M=int(toks[i+1]); i+=2
    K=int(toks[i]); i+=1; depot=int(toks[i]); i+=1
    if M!=0: raise RuntimeError(f'expected Euclidean M=0, got M={M}')
    G=int(toks[i]); i+=1; cust={}
    for _ in range(G):
        n=int(toks[i]); dem=float(toks[i+1]); service=float(toks[i+2]); earliest=float(toks[i+3]); latest=float(toks[i+4]); i+=5
        if n in cust: raise RuntimeError(f'duplicate native customer {n}')
        cust[n]=(dem,service,earliest,latest)
    cap=float(toks[i]); i+=1
    if i!=len(toks): raise RuntimeError(f'unconsumed native tokens: {len(toks)-i}')
    return {'N':N,'M':M,'K':K,'depot':depot,'G':G,'customers':cust,'capacity':cap}

manifest=json.loads((ROOT/'metadata/instance_manifest_subset.json').read_text())
META={}
for m in manifest:
    base=m['family'].replace('/','_')+'_'+m['id']
    m['native_input']=str(DATA/(base+'.txt')); m['coords_csv']=str(DATA/(base+'_coords.csv'))
    seed=DATA/(base+'_v31_tw_feasible_seed.csv'); m['v31_tw_seed_csv']=str(seed) if seed.exists() else None
    raw=RAW/(m['id']+'.vrp'); sol=RAW/(m['id']+'.sol')
    m['parsed']=parse_vrplib(raw)
    assert hashlib.sha256(raw.read_bytes()).hexdigest()==m['raw_vrp_sha256'], f"raw VRP hash drift {m['id']}"
    assert hashlib.sha256(sol.read_bytes()).hexdigest()==m['raw_sol_sha256'], f"raw solution hash drift {m['id']}"
    assert hashlib.sha256(Path(m['native_input']).read_bytes()).hexdigest()==m['input_sha256'], f"input hash drift {m['id']}"
    native=parse_native_euclidean_input(m['native_input'])
    assert native['N']==m['parsed']['dimension'] and native['K']==m['K'] and native['depot']==0
    assert native['G']==m['customers']==m['parsed']['dimension']-1 and abs(native['capacity']-m['capacity'])<1e-9
    assert set(native['customers'])==set(range(1,m['parsed']['dimension']))
    for n,(dem,service,earliest,latest) in native['customers'].items():
        src=n+1
        assert abs(dem-m['parsed']['demands'][src])<1e-9
        exp_service=m['parsed']['service'].get(src,m['parsed']['const_service'])
        assert abs(service-exp_service)<1e-9
        if m['use_time_window']:
            e,l=m['parsed']['time_windows'][src]; assert abs(earliest-e)<1e-9 and abs(latest-l)<1e-9
        else:
            assert earliest==0.0 and latest>=1e11
    with Path(m['coords_csv']).open(newline='') as f: cr=list(csv.DictReader(f))
    assert len(cr)==m['parsed']['dimension']
    assert {int(row['node']) for row in cr}==set(range(m['parsed']['dimension']))
    for row in cr:
        nid=int(row['node']); x,y=m['parsed']['coords'][nid+1]
        assert abs(float(row['x'])-x)<1e-9 and abs(float(row['y'])-y)<1e-9
    ref=reference_common_cost(m['parsed'],parse_sol(sol),m['use_time_window'])
    assert abs(ref-float(m['reference_cost_common']))<1e-7, f"reference common metric mismatch {m['id']}: {ref}"
    # Final audited scoring contract: distance only; route count is diagnostic.
    assert m.get('comparison_rule')=='distance', f"stale comparison rule {m['id']}"
    if m['use_time_window']:
        assert m.get('vrptw_objective')=='distance_only_dimacs'
        assert m['truncate_1_decimal'] and not m['round_distance'] and m['time_equals_distance']
        assert m['distance_convention']=='dimacs_truncate_1_decimal'
        assert m.get('k_explicit')==m['K'], f"declared fleet drift {m['id']}"
    else:
        assert m['distance_convention']=='nearest_integer_euc2d' and m['round_distance'] and not m['truncate_1_decimal']
        assert m['K']==m['customers'], f"Set-X cap must remain nonrestrictive {m['id']}"
    # Re-audit the published reference route with the same evaluator, not just its stored cost.
    ref_routes=[[0]+rr+[0] for rr in parse_sol(sol)['routes']]
    ref_val=validate_euclidean(m,ref_routes)
    assert ref_val['valid'], f"reference route infeasible under common evaluator: {m['id']}"
    assert ref_val['used_routes']==int(m['reference_routes']), f"reference route-count drift {m['id']}"
    assert abs(ref_val['common_distance']-float(m['reference_cost_common']))<1e-7
    META[(m['family'],m['id'])]=m

PARENT_RECEIPT=json.loads((ROOT/'metadata/recovery_parent_invalid18_receipt.json').read_text())
assert PARENT_RECEIPT['target_count']==18
assert PARENT_RECEIPT['parent_zip_sha256']=='e246e58dd948c6df279d9492e517531b8e1d681375c1e75a259c9d6ae656d938'
assert all(t['external_timeout'] and not t['setup_timeout'] and not t['process_ok'] and not t['route_valid'] for t in PARENT_RECEIPT['targets'])
EXPECTED_PARENT_SEMANTIC_SHA={(t['family'],t['instance'],t['solver'],int(t['seed'])):t['semantic_command_sha256'] for t in PARENT_RECEIPT['targets']}
EXPECTED_PARENT_TARGETS=[(t['family'],t['instance'],t['solver'],int(t['seed']),int(t['budget_s'])) for t in PARENT_RECEIPT['targets']]

source_paths={'legacy':SRC/'solver_legacy_STRICT_FAIR_V6.cpp','v31':SRC/'solver_v31_STRICT_FAIR_V6.cpp'}
compile_receipt={'compiler':subprocess.check_output(['g++','--version'],text=True).splitlines()[0],'platform':platform.platform(),'cpu_count':os.cpu_count(),'sources':{}}
for name,sp in source_paths.items():
    sha=hashlib.sha256(sp.read_bytes()).hexdigest(); assert sha==EXPECTED_BENCH_SOURCE_SHA[name]
    out=BIN/f'solver_{name}'; cmd=['g++','-O3','-DNDEBUG','-std=c++17','-pthread',str(sp),'-o',str(out)]
    print('Compiling',name,'...'); subprocess.run(cmd,check=True)
    compile_receipt['sources'][name]={'benchmark_source_sha256':sha,'parent_original_source_sha256':EXPECTED_ORIGINAL_SOURCE_SHA[name],'binary_sha256':hashlib.sha256(out.read_bytes()).hexdigest(),'command':cmd}
(RESULTS/'compile_receipt.json').write_text(json.dumps(compile_receipt,indent=2))
print('PASS: mini payload, raw/public hashes, native conversion contracts, six exact instances, reference feasibility/cost, final distance-only contracts, parent failure receipt, and both frozen sources verified.')


In [ ]:
# @title 2/3 — Run exactly the 18 previously watchdog-killed cells once
import time, shlex, datetime

TARGETS=[
 ('Gehring-Homberger-200','C2_2_1','legacy',2777,120),
 ('Gehring-Homberger-200','R1_2_1','v31',777,120),('Gehring-Homberger-200','R1_2_1','v31',1777,120),('Gehring-Homberger-200','R1_2_1','v31',2777,120),
 ('Gehring-Homberger-200','R2_2_1','v31',777,120),('Gehring-Homberger-200','R2_2_1','v31',1777,120),('Gehring-Homberger-200','R2_2_1','v31',2777,120),
 ('Gehring-Homberger-200','RC2_2_1','v31',1777,120),('Gehring-Homberger-200','RC2_2_1','v31',2777,120),
 ('CVRPLIB-X','X-n200-k36','legacy',777,120),('CVRPLIB-X','X-n200-k36','legacy',1777,120),('CVRPLIB-X','X-n200-k36','legacy',2777,120),
 ('CVRPLIB-X','X-n303-k21','legacy',777,180),('CVRPLIB-X','X-n303-k21','v31',777,180),
 ('CVRPLIB-X','X-n303-k21','v31',1777,180),('CVRPLIB-X','X-n303-k21','legacy',1777,180),
 ('CVRPLIB-X','X-n303-k21','legacy',2777,180),('CVRPLIB-X','X-n303-k21','v31',2777,180),
]
EXPECTED_KEYS={(f,i,s,int(seed)) for f,i,s,seed,b in TARGETS}; assert len(EXPECTED_KEYS)==18
assert TARGETS==EXPECTED_PARENT_TARGETS, 'target list/order drift from parent invalid-18 receipt'

CLOCK_MARKERS=('[BudgetScope] solve clock started before warm start','[STRICT_FAIR_V6]')
SETUP_WATCHDOG=lambda budget:max(180.0,0.75*budget+60.0)
POST_DEADLINE_EXIT_GRACE_S=120.0
POLL_S=0.25

def command_for(meta,solver,seed,budget,run_dir):
    cmd=[str(BIN/f'solver_{solver}'),'--seed',str(seed),'--instances','6','--objective_mode','distance',
         '--time_limit','10.0','--hard_time_limit_s',str(budget),'--no_improve_stop_s','0',
         '--use_time_window',str(meta['use_time_window']).lower(),'--allow_multi_trip','false','--max_trips_per_truck','1',
         '--metric','euclidean','--route_start_time_s',str(float(meta['route_start_s'])),'--log_convergence','false',
         '--run_summary_csv',str(run_dir/'run_summary.csv'),'--coords_csv',meta['coords_csv'],
         '--euclidean_round_distance',str(meta['round_distance']).lower(),'--euclidean_truncate_1_decimal',str(meta['truncate_1_decimal']).lower(),
         '--euclidean_time_equals_distance',str(meta['time_equals_distance']).lower(),'--use_hgs_lite','false','--euclidean_hgs_only','false']
    if solver=='v31':
        cmd += ['--fill_time_budget','true','--use_early_throughput_lns','true']
        if meta.get('use_time_window') and meta.get('v31_tw_seed_csv'):
            cmd += ['--use_cluster_warm_start','true','--clusters_csv',meta['v31_tw_seed_csv']]
    return cmd

def semantic_cmd(cmd):
    ignore_values={'--run_summary_csv','--coords_csv','--clusters_csv'}; out=[]; i=1
    while i<len(cmd):
        k=cmd[i]
        if k in ignore_values:
            out += [k,Path(cmd[i+1]).name]; i+=2
        else:
            out.append(k); i+=1
            if i<len(cmd) and not cmd[i].startswith('--'): out.append(cmd[i]); i+=1
    return out

# Preflight every command before starting the one-shot recovery experiment.
for fam,inst,solver,seed,budget in TARGETS:
    sem=semantic_cmd(command_for(META[(fam,inst)],solver,seed,budget,Path('/tmp/RECOVERY_PREFLIGHT')))
    sem_sha=hashlib.sha256(json.dumps(sem,separators=(',',':')).encode()).hexdigest()
    assert sem_sha==EXPECTED_PARENT_SEMANTIC_SHA[(fam,inst,solver,int(seed))], f"command semantics drift: {fam}/{inst}/{solver}/{seed}"
print('PASS: 18/18 recovery commands reproduce parent semantics exactly.')

lock=ROOT/'RECOVERY_ATTEMPT_STARTED.lock'
if lock.exists():
    raise RuntimeError('Recovery cell already started in this runtime. Restart runtime before intentionally starting a new recovery experiment.')
lock.write_text(json.dumps({'started_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'targets':TARGETS},indent=2))

def run_target(family,instance,solver,seed,budget):
    meta=META[(family,instance)]; tag=f'{family}__{instance}__seed{seed}__{solver}'.replace('/','_')
    rd=RUNS/tag; rd.mkdir(parents=True,exist_ok=False)
    stdout=rd/'stdout.txt'; stderr=rd/'stderr.txt'; cmd=command_for(meta,solver,seed,budget,rd)
    (rd/'command.txt').write_text(' '.join(shlex.quote(x) for x in cmd)+' < '+shlex.quote(meta['native_input'])+'\n')
    semantic=semantic_cmd(cmd); sem_sha=hashlib.sha256(json.dumps(semantic,separators=(',',':')).encode()).hexdigest()
    assert sem_sha==EXPECTED_PARENT_SEMANTIC_SHA[(family,instance,solver,int(seed))]
    scan_pos=0; scan_buf=''; algo_t0=None; clock_marker_observed=None; timed_out=False; setup_timeout=False; process_t0=time.perf_counter()
    with open(meta['native_input'],'rb') as fin, stdout.open('wb') as fout, stderr.open('wb') as ferr:
        proc=subprocess.Popen(cmd,stdin=fin,stdout=fout,stderr=ferr,cwd=rd)
        while proc.poll() is None:
            try:
                with stderr.open('rb') as sf:
                    sf.seek(scan_pos); new=sf.read(); scan_pos=sf.tell()
                if new:
                    scan_buf=(scan_buf+new.decode('utf-8','replace'))[-32768:]
                    if algo_t0 is None:
                        hit=next((m for m in CLOCK_MARKERS if m in scan_buf),None)
                        if hit is not None: clock_marker_observed=hit; algo_t0=time.perf_counter()
            except: pass
            now=time.perf_counter()
            if algo_t0 is None:
                if now-process_t0>SETUP_WATCHDOG(budget): setup_timeout=True; timed_out=True; proc.kill(); break
            elif now-algo_t0>budget+POST_DEADLINE_EXIT_GRACE_S:
                timed_out=True; proc.kill(); break
            time.sleep(POLL_S)
        rc=proc.wait() if proc.poll() is not None else -9
    process_wall=time.perf_counter()-process_t0; algo_wall=(time.perf_counter()-algo_t0) if algo_t0 else None
    text=stdout.read_text(errors='replace'); routes=parse_routes(text); val=validate_euclidean(meta,routes)
    process_ok=(rc==0 and not timed_out and algo_t0 is not None); valid=bool(process_ok and val['valid'])
    ref=float(meta['reference_cost_common']); gap=100.0*(val['common_distance']-ref)/ref if valid else None
    result={'family':family,'instance':instance,'solver':solver,'seed':seed,'budget_s':budget,'workers':6,
            'benchmark_solver_sha256':EXPECTED_BENCH_SOURCE_SHA[solver],'original_solver_sha256':EXPECTED_ORIGINAL_SOURCE_SHA[solver],
            'input_sha256':meta['input_sha256'],'semantic_command_sha256':sem_sha,'semantic_command':semantic,
            'returncode':rc,'external_timeout':timed_out,'setup_timeout':setup_timeout,'process_ok':process_ok,
            'route_valid':val['valid'],'valid':valid,'comparison_rule':'distance','common_distance':val['common_distance'],'used_routes':val['used_routes'],
            'lateness':val['lateness'],'reference_cost_common':ref,'reference_routes':int(meta['reference_routes']),
            'fleet_delta_vs_reference':(val['used_routes']-int(meta['reference_routes'])) if valid else None,
            'gap_to_reference_pct':gap,'clock_marker_observed':clock_marker_observed,
            'post_deadline_exit_grace_s':POST_DEADLINE_EXIT_GRACE_S,
            'process_wall_s':process_wall,'algorithm_clock_observed_s':algo_wall,
            'stdout_sha256':hashlib.sha256(stdout.read_bytes()).hexdigest(),'stderr_sha256':hashlib.sha256(stderr.read_bytes()).hexdigest(),
            'final_route_sha256':hashlib.sha256('|'.join(','.join(str(x) for x in r) for r in active_routes(routes,0)).encode()).hexdigest() if routes else None,
            'runtime_comparable_to_parent':False,'noncontemporaneous_recovery':True,'validation':val}
    (rd/'common_validation.json').write_text(json.dumps(val,indent=2)); (rd/'recovery_result.json').write_text(json.dumps(result,indent=2))
    shown=f"{val['common_distance']:.3f}" if routes else 'NO ROUTE'
    print(f"{family:24s} {instance:10s} {solver:6s} seed={seed} -> valid={valid} common={shown} routes={val['used_routes']} rc={rc}")
    return result

RECOVERY_RESULTS=[run_target(*t) for t in TARGETS]
assert {(r['family'],r['instance'],r['solver'],int(r['seed'])) for r in RECOVERY_RESULTS}==EXPECTED_KEYS
print(f"Finished 18/18 fixed attempts. Valid recoveries: {sum(bool(r['valid']) for r in RECOVERY_RESULTS)}/18")


In [ ]:
# @title 3/3 — Audit, summarize, ZIP recovery evidence, download
import pandas as pd, zipfile, datetime
rows=[json.loads(p.read_text()) for p in sorted(RUNS.glob('*/recovery_result.json'))]
assert len(rows)==18
keys={(r['family'],r['instance'],r['solver'],int(r['seed'])) for r in rows}; assert keys==EXPECTED_KEYS
flat=[{k:v for k,v in r.items() if k not in ('validation','semantic_command')} for r in rows]
df=pd.DataFrame(flat).sort_values(['family','instance','solver','seed']).reset_index(drop=True)
df.to_csv(RESULTS/'recovery_runs.csv',index=False)
summary_rows=[]
for (fam,inst,solver),g in df.groupby(['family','instance','solver'],dropna=False,sort=False):
    gv=g[g['valid'].astype(bool)].copy()
    vals=gv['common_distance'].dropna().astype(float)
    summary_rows.append({'family':fam,'instance':inst,'solver':solver,'attempts':len(g),'valid_runs':len(gv),
                         'best_common_distance':float(vals.min()) if len(vals) else float('nan'),
                         'mean_common_distance':float(vals.mean()) if len(vals) else float('nan'),
                         'reference_cost_common':float(g['reference_cost_common'].iloc[0]),
                         'comparison_rule':'distance','runtime_comparable_to_parent':False})
grp=pd.DataFrame(summary_rows)
grp.to_csv(RESULTS/'recovery_summary.csv',index=False)
df[~df['valid'].astype(bool)].to_csv(RESULTS/'unresolved_invalid.csv',index=False)
receipt={'recovery_protocol':'V7_LIGHT_TARGETED_RECOVERY_AUDIT3','created_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
         'target_count':18,'valid_recovered':int(df['valid'].astype(bool).sum()),'unresolved':int((~df['valid'].astype(bool)).sum()),
         'post_deadline_exit_grace_s':POST_DEADLINE_EXIT_GRACE_S,'comparison_rule':'distance','reference_routes_role':'diagnostic_only','runtime_comparison_permitted':False,
         'target_keys':sorted([list(x) for x in EXPECTED_KEYS]),'payload_sha256':PAYLOAD_SHA256,
         'benchmark_source_sha256':EXPECTED_BENCH_SOURCE_SHA,'original_parent_source_sha256':EXPECTED_ORIGINAL_SOURCE_SHA,
         'parent_invalid18_receipt_sha256':hashlib.sha256((ROOT/'metadata/recovery_parent_invalid18_receipt.json').read_bytes()).hexdigest(),
         'note':'Recovery outputs are feasibility/solution-quality replicates; do not silently substitute them into strict original-batch paired runtime claims.'}
(RESULTS/'recovery_manifest.json').write_text(json.dumps(receipt,indent=2))
zip_path=Path('/content/VRP_V7_LIGHT_TARGETED_RECOVERY_RESULTS.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=9) as z:
    for base in (RESULTS,RUNS,SRC,DATA,RAW,ROOT/'metadata'):
        for p in base.rglob('*'):
            if p.is_file(): z.write(p,p.relative_to(ROOT))
sha=hashlib.sha256(zip_path.read_bytes()).hexdigest()
print('\nFINAL RECOVERY SUMMARY')
print(df[['family','instance','solver','seed','valid','common_distance','used_routes','gap_to_reference_pct']].to_string(index=False))
print(f"\nZIP: {zip_path} size={zip_path.stat().st_size/1024:.1f} KiB sha256={sha}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
